# 3 N-BEATS: putting the decomposition *inside* the architecture

Suppose you computed a trend slope and a set of FFT band energies by hand and bolted them onto
the model as a second input stream. Measured, the model ignores them — and the diagnosis is that
a linear layer over the raw window can already compute a trend slope, since it is a weighted
sum. You handed it a column it already had.

That diagnosis suggests an obvious next move. If trend and seasonality are the right vocabulary
for a timeseries, do not compute them outside the model and pass them in. **Make them the shape
the model is allowed to predict in.**

That is N-BEATS (Oreshkin et al., 2020). Instead of a network that outputs 24 free numbers, you
get a network that outputs *coefficients of a basis* — a polynomial for trend, a Fourier series
for seasonality — and the forecast is whatever those coefficients build. The architecture cannot
predict an arbitrary squiggle. It can only predict a trend plus a seasonal pattern, and it has to
say how much of each.

**You build this one yourself.** The libraries this course uses ship no pretrained N-BEATS the
way torchvision ships a pretrained `resnet18`, and N-BEATS is usually trained for one dataset
and one horizon. (`neuralforecast` has an implementation, built on pytorch-lightning, whose
top-level model takes a Lightning batch dict.) So we build it — about forty lines, mirroring
[`neuralforecast/models/nbeats.py`](https://github.com/Nixtla/neuralforecast/blob/main/neuralforecast/models/nbeats.py),
close enough that you can read theirs afterwards and recognise every part.

In [ ]:
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from loguru import logger
from notebooktester import param
from solar_10_minutes import load_solar
from sunspot_daily import PERIOD as SUN_PERIOD
from sunspot_daily import load_sunspot_daily
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings
from windowing import window_index

W, H = 288, 144  # two days of history, one day of horizon, at 10-minute resolution
# one day, in 10-minute steps -- the seasonality we believe is in this data
PERIOD = 144
# 10 harmonics of one day: the shortest wave lasts 2.4 hours, short enough to bend
# around sunrise and sunset
HARMONICS = 10
SEEDS = param((0, 1, 2), test=(0,))
N_SERIES = param(20, test=2)  # sites of the 137, pooled
# a ceiling, not a choice: early stopping ends each run where its own validation loss
# stops falling (3.4), and a run that reaches the ceiling shows it as stopped = 40
MAX_EPOCHS = param(40, test=1)
# the N-BEATS validation loss bumps up and down from epoch to epoch by about as much as it
# still improves (see the loss curves in 3.4), so a stop needs five epochs without a new best,
# not one bump
PATIENCE = 5
logger.remove()  # Trainer logs every epoch; keep warnings only
logger.add(sys.stderr, level="WARNING")
torch.manual_seed(0)

## 3.1 A series with something for a Fourier basis to find

A seasonal basis is only worth having if there is seasonality. Not every series has
some at a period you can window over — `3.6` comes back to that question directly.

So this notebook opens on **solar power production** — ten-minute readings from 137
sites in the Monash `solar_10_minutes` set, where the sun's daily arc is most of the
story: every site sits at exactly zero for roughly half of each day, night after
night, whatever the weather is doing to the daytime hump.

In [ ]:
def split_scale_window(series: list[torch.Tensor], stride: int = 6):
    # 60/20/20 in time, per series, scaled on that series' own training span, then windowed.
    # Also returns, per window, the step of the year at which its forecast starts, so a score
    # can be broken down by month (3.4).
    parts = {"train": [[], [], []], "val": [[], [], []], "test": [[], [], []]}
    for s in series:
        n = len(s)
        cuts = {
            "train": (0, int(0.6 * n)),
            "val": (int(0.6 * n), int(0.8 * n)),
            "test": (int(0.8 * n), n),
        }
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            index = window_index(len(v), W + H, stride)
            win = v[index]
            parts[split][0].append(win[:, :W])
            parts[split][1].append(win[:, W:])
            parts[split][2].append(a + index[:, W])
    windows = {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}
    origins = {k: torch.cat(v[2]) for k, v in parts.items()}
    return windows, origins


sites = load_solar(W, H, n_series=N_SERIES)
power, origins = split_scale_window(sites)
for split, (x, y) in power.items():
    print(f"{split:5s} x {tuple(x.shape)}  y {tuple(y.shape)}")

In [ ]:
train_x = power["train"][0]
sample = train_x[0]

n = len(sample)
z = (sample - sample.mean()) * torch.hann_window(n, periodic=False)
spec = torch.fft.rfft(z).abs() ** 2
steps = 1.0 / torch.fft.rfftfreq(n, d=1.0)[1:]
hours = steps * (10 / 60)  # 10-minute steps -> hours, for a readable axis

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(sample, lw=0.8)
ax[0].set_title("one 2-day window of solar power (288 ten-minute steps)")
ax[0].set_xlabel("10-minute steps")
ax[1].loglog(hours, spec[1:], lw=0.8)
ax[1].axvline(24, color="crimson", ls="--", lw=1)
ax[1].text(24, spec[1:].max(), " 24h", color="crimson", va="top", fontsize=9)
ax[1].set_xlabel("period (hours)")
ax[1].set_title("its spectrum")
plt.tight_layout()

A spike at 24 hours you can see without squinting. That is the thing a Fourier basis is for.

## 3.2 A basis is a fixed set of shapes; the network picks the mix

Every N-BEATS block does the same two steps:

1. an ordinary MLP maps the input window to a short vector of coefficients, **theta**;
2. a **basis** turns theta into two curves — a *backcast* (its reconstruction of the input) and a
   *forecast* (its prediction of the future).

The basis has **no learnable parameters**. It is a fixed matrix of shapes, and all the network
does is decide how much of each shape to use. Swap the basis and you change what the block is
*able* to say.

The trend basis is powers of normalised time: a constant, a ramp, a curve, a bend.

In [ ]:
class TrendBasis(nn.Module):
    # theta -> a low-order polynomial in normalised time, for backcast and forecast

    backcast_basis: torch.Tensor
    forecast_basis: torch.Tensor

    def __init__(self, degree: int, backcast_size: int, forecast_size: int):
        super().__init__()
        self.size = degree + 1
        powers = lambda n: torch.stack(
            [torch.linspace(0, 1, n) ** p for p in range(self.size)]
        )
        self.register_buffer("backcast_basis", powers(backcast_size))  # (size, W)
        self.register_buffer("forecast_basis", powers(forecast_size))  # (size, H)

    def forward(self, theta):
        backcast = torch.einsum("bp,pt->bt", theta[:, : self.size], self.backcast_basis)
        forecast = torch.einsum("bp,pt->bt", theta[:, self.size :], self.forecast_basis)
        return backcast, forecast


trend = TrendBasis(degree=3, backcast_size=W, forecast_size=H)
fig, ax = plt.subplots(figsize=(9, 2.5))
for p, row in enumerate(trend.forecast_basis):
    ax.plot(row, label=f"t^{p}")
ax.set_title("the trend basis: four shapes, no parameters")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
print(
    f"theta for this basis has {2 * trend.size} numbers "
    f"({trend.size} for the backcast, {trend.size} for the forecast)"
)

And the seasonal basis is a Fourier series: sines and cosines at harmonics of a period you name.

Look at what that means. Computing FFT *band energies* and handing them to the model as inputs
puts the spectrum on the input side. Here the same sines and cosines are the **output** vocabulary — the model
does not receive a spectrum, it predicts *in* one. Same mathematics, opposite direction.

In [ ]:
class SeasonalityBasis(nn.Module):
    # theta -> a Fourier series at harmonics of a known period

    backcast_basis: torch.Tensor
    forecast_basis: torch.Tensor

    def __init__(
        self, harmonics: int, period: int, backcast_size: int, forecast_size: int
    ):
        super().__init__()
        self.size = 2 * harmonics
        # the backcast grid runs up to the forecast origin, the forecast grid starts at it,
        # so the two are one continuous timeline and the phase carries across the boundary
        for name, n, offset in (
            ("backcast", backcast_size, -backcast_size),
            ("forecast", forecast_size, 0),
        ):
            t = (torch.arange(n, dtype=torch.float32) + offset) / period
            rows = [
                f(2 * np.pi * k * t)
                for k in range(1, harmonics + 1)
                for f in (torch.cos, torch.sin)
            ]
            self.register_buffer(f"{name}_basis", torch.stack(rows))

    def forward(self, theta):
        backcast = torch.einsum("bp,pt->bt", theta[:, : self.size], self.backcast_basis)
        forecast = torch.einsum("bp,pt->bt", theta[:, self.size :], self.forecast_basis)
        return backcast, forecast


seasonal = SeasonalityBasis(
    harmonics=HARMONICS, period=PERIOD, backcast_size=W, forecast_size=H
)

fig, ax = plt.subplots(figsize=(9, 2.5))
for i in range(4):
    ax.plot(seasonal.forecast_basis[i], lw=1)
ax.set_title(
    f"the seasonality basis: harmonics of a {PERIOD}-step (24h) period (first four shown)"
)
plt.tight_layout()

# a basis is only trustworthy if it can express the thing it claims to
target = torch.cos(2 * np.pi * torch.arange(H, dtype=torch.float32) / PERIOD)
theta = torch.zeros(1, 2 * seasonal.size)
theta[0, seasonal.size] = 1.0  # one unit of the first cosine, nothing else
_, reconstructed = seasonal(theta)
print(
    f"reconstructs a pure {PERIOD}-step cosine exactly: "
    f"max error {float((reconstructed[0] - target).abs().max()):.2e}"
)

## 3.3 The block, and the stacking that makes it work

A block is the MLP plus a basis. The interesting part is what happens *between* blocks.

Each block emits a backcast — its own account of the input — and that account is **subtracted from
the input before the next block sees it**. Block two therefore works on what block one could not
explain. Meanwhile the forecasts are **added up**. Oreshkin et al. call this *doubly residual
stacking*, and it is the whole mechanism: subtract backwards, accumulate forwards.

Give the first block a trend basis and the second a seasonal one. Both blocks are trained
together on one loss, so nothing forces the first block to take "the trend" in any deeper
sense. What the bases fix is what each block *can* say: a cubic over the two-day input
cannot draw its two daily humps, and a Fourier series with no constant term cannot draw a
level. Whatever the
trained model does, its forecast is split into a part of the first kind and a part of the
second.

One more detail in `NBeats.forward` below: the forecast does not start at zero. It starts at
the **last observed value**, repeated over the horizon (the naive forecast of 3.4), and the
blocks add their forecasts to that start. So the blocks only learn a correction to "repeat the
last value". 3.5 has to account for that start when it asks what each stack contributed.

In [ ]:
class IdentityBasis(nn.Module):
    # theta IS the curve - no structure imposed. This is "generic" N-BEATS.

    def __init__(self, backcast_size: int, forecast_size: int):
        super().__init__()
        self.size = backcast_size

    def forward(self, theta):
        return theta[:, : self.size], theta[:, self.size :]


class NBeatsBlock(nn.Module):
    def __init__(
        self,
        basis,
        input_size: int,
        theta_size: int,
        hidden: int = 256,
        layers: int = 3,
    ):
        super().__init__()
        mlp, d = [], input_size
        for _ in range(layers):
            mlp += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        self.mlp = nn.Sequential(*mlp)
        self.theta = nn.Linear(hidden, theta_size)
        self.basis = basis

    def forward(self, x):
        return self.basis(self.theta(self.mlp(x)))


class NBeats(nn.Module):
    def __init__(self, blocks: list[NBeatsBlock]):
        super().__init__()
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x, per_block: bool = False):
        residual = x
        forecast = x[:, -1:].expand(-1, H).clone()  # start at the naive forecast
        contributions = []
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast  # <- subtract what was explained
            forecast = forecast + block_forecast  # <- accumulate what was predicted
            contributions.append(block_forecast)
        return (forecast, contributions) if per_block else forecast


def interpretable(degree=3, harmonics=HARMONICS, period=PERIOD, hidden=256):
    t = TrendBasis(degree, W, H)
    s = SeasonalityBasis(harmonics, period, W, H)
    return NBeats(
        [NBeatsBlock(t, W, 2 * t.size, hidden), NBeatsBlock(s, W, 2 * s.size, hidden)]
    )


def generic(n_blocks=2, hidden=256):
    return NBeats(
        [NBeatsBlock(IdentityBasis(W, H), W, W + H, hidden) for _ in range(n_blocks)]
    )


model = interpretable()
out, parts = model(train_x[:4], per_block=True)
print(
    f"forecast {tuple(out.shape)}, from {len(parts)} stacks of {tuple(parts[0].shape)}"
)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 3.4 Baselines first

The free baselines first, then the cheap trained model, then the expensive one. With a real
24-hour cycle, seasonal naive should be the stronger of the two free baselines. Check that
before going further; 3.6 runs the same baselines on a series without a cycle at this period.

**How long each model trains.** No epoch count is picked here. Every model trains with early
stopping, as in lesson 1 (`1_representation_and_head/03_training`): after each epoch the
`Trainer` computes the loss on the validation split, keeps the weights of the epoch with the
lowest one, and stops once `PATIENCE` epochs in a row bring no new lowest. A linear layer and a
three-layer MLP stack converge at different speeds, so each one stops where its own validation
loss says. That leaves two jobs for three splits: **validation** decides when to stop, and
**test**, which neither training nor stopping ever sees, is the split every score in this
notebook is reported on.

Before the next cell runs, write down two expectations: will the N-BEATS models beat the linear
layer, and will any trained model beat seasonal naive, which only copies yesterday?

In [ ]:
def mae(y, yhat):
    return float((y - yhat).abs().mean())


def spread(scores: list[float]) -> float:
    # sample std over seeds (ddof=1); undefined for a single seed
    return float(np.std(scores, ddof=1)) if len(scores) > 1 else float("nan")


def fit(make_model, data, seeds=SEEDS, max_epochs=MAX_EPOCHS, lr=1e-3, batch_size=128):
    # One Trainer per seed. Early stopping watches the validation loss (MSE) and hands back
    # the model of its best epoch; test is scored by hand afterwards on that model, because
    # Trainer never sees the test split.
    xtr, ytr = data["train"]
    xva, yva = data["val"]
    xte, yte = data["test"]
    validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))
    runs = {"test": [], "val": [], "best": [], "stopped": [], "secs": [], "curves": []}
    models = []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        trainloader = DataLoader(
            TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
        )
        settings = TrainerSettings(
            epochs=max_epochs,
            metrics=[MAE()],
            logdir=Path("modellogs").resolve(),
            train_steps=len(trainloader),
            valid_steps=len(validloader),
            optimizer_kwargs={"lr": lr},
            earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
        )
        curve = []  # (train loss, validation loss) per epoch, for the loss curves

        def record(epoch, train_loss, valid_loss, metrics, curve=curve):
            curve.append((train_loss, valid_loss))

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=nn.MSELoss(),
            optimizer=torch.optim.Adam,
            traindataloader=trainloader,
            validdataloader=validloader,
            reporters=[record],
            progress=False,
        )
        t0 = time.time()
        result = trainer.loop()  # trainer.model now holds the weights of the best epoch
        runs["secs"].append(time.time() - t0)
        runs["val"].append(result.metric_dict["MAE"])
        runs["best"].append(result.epoch + 1)  # Trainer counts epochs from 0
        runs["stopped"].append(result.stopped_epoch + 1)
        runs["curves"].append(curve)
        models.append(model)
        model.eval()
        with torch.no_grad():
            runs["test"].append(mae(yte, model(xte)))
    return {
        "mae": float(np.mean(runs["test"])),
        "std": spread(runs["test"]),
        "val": float(np.mean(runs["val"])),
        "best": float(np.mean(runs["best"])),
        "stopped": float(np.mean(runs["stopped"])),
        "secs": float(np.mean(runs["secs"])),
        "params": sum(p.numel() for p in model.parameters()),
        "curves": runs["curves"],
        "models": models,  # one per seed, each with the weights of its best epoch
        "model": model,
    }


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):
        return self.fc(x)


def free(forecast, data) -> dict[str, Any]:
    # a baseline with nothing to train: no seeds, no epochs, no parameters
    (xva, yva), (xte, yte) = data["val"], data["test"]
    nan = float("nan")
    return {
        "mae": mae(yte, forecast(xte)),
        "std": 0.0,
        "val": mae(yva, forecast(xva)),
        "best": nan,
        "stopped": nan,
        "secs": 0.0,
        "params": 0,
    }


def naive(x):
    return x[:, -1:].expand(-1, H)


def seasonal_naive(x, period=PERIOD):
    # repeat the last `period` steps until the horizon is full
    return x[:, -period:].repeat(1, -(-H // period))[:, :H]


results: dict[str, dict[str, Any]] = {
    "naive (repeat last)": free(naive, power),
    "seasonal naive (24h)": free(seasonal_naive, power),
}
for name, r in results.items():
    print(f"{name:24s} test MAE {r['mae']:.4f}   val MAE {r['val']:.4f}")
xte, yte = power["test"]

In [ ]:
for name, make in (
    ("linear", Linear),
    ("N-BEATS generic", generic),
    ("N-BEATS interpretable", interpretable),
):
    r = fit(make, power)
    results[name] = r
    print(
        f"{name:24s} test MAE {r['mae']:.4f} +/- {r['std']:.4f}   val MAE {r['val']:.4f}   "
        f"best epoch {r['best']:4.1f}   stopped {r['stopped']:4.1f}   {r['secs']:5.1f}s"
    )

In [ ]:
def table(results):
    head = f"{'model':24s} {'test MAE':>9} {'+/-':>7} {'val MAE':>8} {'best ep':>8} {'stopped':>8} {'params':>9} {'secs':>7}"
    print(head)
    print("-" * len(head))
    for name in sorted(results, key=lambda k: results[k]["mae"]):
        r = results[name]
        ep = lambda v: f"{v:8.1f}" if v == v else f"{'-':>8}"
        print(
            f"{name:24s} {r['mae']:9.4f} {r['std']:7.4f} {r['val']:8.4f} {ep(r['best'])} {ep(r['stopped'])} "
            f"{r['params']:9,} {r['secs']:7.1f}"
        )


table(results)

In [ ]:
def plot_curves(results, names, title):
    fig, axes = plt.subplots(1, len(names), figsize=(4.5 * len(names), 3), sharey=True)
    for ax, name in zip(axes, names):
        for seed, curve in zip(SEEDS, results[name]["curves"]):
            train, valid = np.array(curve).T
            epochs = np.arange(1, len(curve) + 1)
            (line,) = ax.plot(epochs, valid, lw=1.5, label=f"seed {seed} validation")
            ax.plot(epochs, train, lw=0.8, ls=":", color=line.get_color())
            best = int(np.argmin(valid))
            ax.plot(epochs[best], valid[best], "o", color=line.get_color())
        ax.set_title(name)
        ax.set_xlabel("epoch")
    axes[0].set_ylabel("MSE loss (scaled units)")
    axes[-1].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()


trained = ["linear", "N-BEATS generic", "N-BEATS interpretable"]
plot_curves(
    results,
    trained,
    "validation loss per epoch (solid), training loss (dotted); the dot is the epoch kept",
)

**Read the test column; validation is where each run chose to stop.** The ± is the sample
standard deviation over the three seeds. It measures how much training luck moves a score,
not how much the score would move on another stretch of test data: the test windows overlap,
come from 20 sites in one region and cover one autumn.

**N-BEATS beats the linear layer, by far more than the seed spread.** On test, generic N-BEATS
scores 0.2508 ± 0.0013 and linear 0.3022 ± 0.0062: a gap of 0.051, about eight times the larger
of the two seed stds. The interpretable variant lands at 0.2571 ± 0.0070, level with generic
(the gap of 0.006 is about one seed std). The two N-BEATS variants differ in more than the
basis: generic has 633,184 parameters and emits a theta of 432 numbers per block (one per
backcast and forecast step), interpretable has 423,472 and emits 8 (trend) and 40
(seasonality). Validation gives the same order (0.2246 and 0.2239 against 0.2502).

**Each model trained as long as its own validation loss asked.** Linear kept its weights of
epoch 6.3 on average and stopped at 11.3; generic kept epoch 3.7 and stopped at 8.7;
interpretable kept 5.3 and stopped at 10.3. The loss curves show why so early: the validation
loss of all three is flat from the second or third epoch on, while the training loss of both
N-BEATS models keeps falling. More epochs fit the training months better and the validation
months no better. So none of the three was still learning when the comparison was made.

Two more things in the curves. The validation loss sits *below* the training loss for every
model: the training loss is averaged over an epoch in which the weights are still changing,
and the validation months are easier months to forecast (the per-month table below shows
seasonal naive itself scoring 0.216 to 0.240 in August to October, against 0.251 to 0.326 in
January to May). And the linear layer's two curves are flat from epoch 2 with no gap opening
between them: it is not overfitting a small training set, it is limited by what one linear
map can express.

**No trained model beats seasonal naive on test.** Copying yesterday scores 0.2201; the best
trained model, 0.2508. On validation the two N-BEATS models edge just past it (0.2246 and
0.2239 against 0.2274), so the two splits disagree. The cut is in time, over one calendar
year: training covers January to early August, validation early August to mid October, test
mid October to December. That is the honest setting for a forecast (you train on the past and
forecast the future), and it means the test score belongs to one stretch of one autumn. The
table below breaks every score down by the month in which the forecast starts.

**Per month.** Train months are scored on windows the models were trained on, so read them as
in-sample; validation and test months are not. Write down first: in which months do you expect
the trained models to lose to seasonal naive?

In [ ]:
def month_of(steps: torch.Tensor) -> np.ndarray:
    # every series starts on 2006-01-01, one step is 10 minutes
    return (
        pd.Timestamp("2006-01-01") + pd.to_timedelta(steps.numpy() * 10, unit="min")
    ).month


def window_mae(model, x, y):
    # MAE per window, in chunks so the 633k-parameter model does not hold all windows at once
    with torch.no_grad():
        return torch.cat(
            [
                (model(a) - b).abs().mean(dim=1)
                for a, b in zip(x.split(8192), y.split(8192))
            ]
        )


frames = []
for split in ("train", "val", "test"):
    x, y = power[split]
    scores = {"seasonal naive": (seasonal_naive(x) - y).abs().mean(dim=1)}
    for name in trained:
        for m in results[name]["models"]:
            m.eval()
        scores[name] = torch.stack(
            [window_mae(m, x, y) for m in results[name]["models"]]
        ).mean(dim=0)  # mean over seeds
    frame = pd.DataFrame({k: v.numpy() for k, v in scores.items()})
    frame["month"], frame["split"] = month_of(origins[split]), split
    frames.append(frame)
frames = pd.concat(frames)
monthly = frames.groupby(["month", "split"]).mean().round(3)
monthly["windows"] = frames.groupby(["month", "split"]).size()
print(
    "MAE per month of the forecast start, mean over seeds (train months are in-sample)"
)
print(monthly.to_string())

**What the months show.** Generic N-BEATS loses to seasonal naive in every month from October
on: in October's validation windows (0.239 against 0.220) and in all three test months
(October 0.275 against 0.260, November 0.251 against 0.216, December 0.243 against 0.211). It
beats seasonal naive in the validation windows of August and September, and in-sample in
January to March, May and July; it loses narrowly in April and clearly in June and early
August. The linear layer is behind generic N-BEATS in every month.

Look at the seasonal-naive column from top to bottom before you read on. The two N-BEATS
models score 0.243 to 0.257 in November and December, close to their 0.252 to 0.277 in January
and February. Seasonal naive does not stay put: 0.211 to 0.216 in November and December,
against 0.271 to 0.326 in January to March. So the gap on test is mostly the copy rule getting
better, not the models getting worse. The next cell asks why.

### A baseline's score belongs to the weeks it is scored on

Seasonal naive forecasts the next 24 hours as a copy of the last 24. Its error is therefore
exactly the difference between two consecutive 24-hour stretches. That is not a property of the rule alone: in
weeks where one day looks like the next (the same clear sky, or the same overcast, day after
day), copying is nearly right; in weeks where sunny and cloudy days alternate, copying is wrong
every day.

So measure that directly, without any model: for every site and every day, the day's **mean
level** (its average scaled power, which is lower when days are short or cloudy) and the
**day-to-day change**, how far that mean level moved from the day before. Both are in the same
scaled units as the MAE.

Write down first: November and December have short days, like January and February. Do you
expect the day-to-day change to be about the same in those four months, or not? And if it is
not, which way would seasonal naive's error move?

In [ ]:
daily = []
for s in sites:
    n = len(s)
    mu, sigma = (
        s[: int(0.6 * n)].mean(),
        s[: int(0.6 * n)].std(),
    )  # as split_scale_window
    days = ((s - mu) / sigma)[: (n // PERIOD) * PERIOD].reshape(-1, PERIOD)
    level = days.mean(dim=1).numpy()  # the mean level of each day
    daily.append(
        pd.DataFrame(
            {
                "month": (
                    pd.Timestamp("2006-01-01")
                    + pd.to_timedelta(np.arange(len(level)), unit="D")
                ).month,
                "day mean level": level,
                "day-to-day change": np.abs(np.diff(level, prepend=np.nan)),
            }
        )
    )
persistence = pd.concat(daily).groupby("month").mean()
by_month = frames.groupby("month")[
    ["seasonal naive", *trained]
].mean()  # all splits pooled
print(pd.concat([by_month, persistence], axis=1).round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4), sharex=True)
for name in by_month.columns:
    ax[0].plot(
        by_month.index,
        by_month[name],
        marker="o",
        lw=3 if name == "seasonal naive" else 1,
        color="k" if name == "seasonal naive" else None,
        label=name,
    )
ax[1].plot(
    persistence.index,
    persistence["day-to-day change"],
    marker="o",
    color="k",
    lw=3,
    label="day-to-day change of the mean level",
)
ax[1].plot(
    persistence.index,
    persistence["day mean level"],
    marker="o",
    ls="--",
    color="0.5",
    label="day mean level",
)
for a in ax:
    a.axvspan(
        8.2, 10.6, color="tab:orange", alpha=0.12
    )  # validation: early Aug to mid Oct
    a.axvspan(10.6, 12.5, color="tab:red", alpha=0.12)  # test: mid Oct to December
    a.set_xticks(range(1, 13))
    a.set_xlabel("month of 2006 (orange: validation, red: test)")
ax[0].set_ylabel("MAE (scaled power)")
ax[0].set_title("forecast error per month")
ax[0].legend(fontsize=7)
ax[1].set_ylabel("scaled power")
ax[1].set_title("how much one day differs from the day before")
ax[1].legend(fontsize=7)
plt.tight_layout()

**Seasonal naive's error follows the weather's day-to-day change.** The thick black lines in
the two panels move together across the year: the copy rule does worst in January to March,
when the mean level changed by 0.241 to 0.288 from one day to the next, and best in June and
August, when it changed by about 0.11. (August and October pool windows from two splits here;
the table above keeps them apart.)

**November and December were calm weeks, not just short days.** Their mean level (−0.191 and
−0.246) is about that of January and February (−0.231 and −0.187): days as short and dark.
But consecutive days were more alike: a day-to-day change of 0.184 to 0.185, against 0.241 to
0.288. Seasonal naive's error fell with it, to 0.211 to 0.216 against 0.271 to 0.326. The two
N-BEATS models barely moved: 0.243 to 0.257 in November and December, 0.252 to 0.277 in
January and February (and those winter months were in-sample for them, which if anything
flatters them there). A model that forecasts a typical next day from two days of history gains
less from a run of look-alike days than a rule that copies the last one.

So the test verdict of 3.4 reads differently now. The trained models did not meet hard weeks
for themselves; the baseline met easy weeks for itself. A gap to a baseline is a property of
the period it is measured on as much as of the two forecasters. What this notebook cannot say
is whether November and December are calm every year or were calm in 2006: the record holds
one year, so season and that year's weather are the same thing in it.


## 3.5 The payoff: ask the model what it thinks the series is made of

The interpretable model does not just score well here — it can also do something
none of the others in this lesson can. `per_block=True` returns each stack's
contribution separately, so you can plot the trend it found and the seasonality it
found as **two curves that sum to the forecast**.

This is not a visualisation trick bolted on afterwards. The stacks are separate because
the architecture made them separate.

In [ ]:
def parts_of(model, x):
    # The forecast is start + trend stack + seasonality stack (3.3). The start is the last
    # observed value, a level, so it is counted with the trend stack: together they are the
    # level part. The seasonality stack has no constant term, and over a horizon of exactly
    # one period its mean is zero: it is pure shape.
    model.eval()
    with torch.no_grad():
        forecast, (trend_stack, seasonal_stack) = model(x, per_block=True)
    return forecast, x[:, -1:] + trend_stack, seasonal_stack


def movement(part: torch.Tensor) -> float:
    # how much a part moves within the horizon: std over the H steps, mean over windows
    return float(part.std(dim=1).mean())


interp = results["N-BEATS interpretable"]["model"]
forecast, level_part, seasonal_part = parts_of(interp, xte[:512])

picks = [3, 61, 140]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 3.2), sharey=True)
for ax, i in zip(axes, picks):
    ax.plot(range(-PERIOD, 0), xte[i, -PERIOD:], color="0.6", lw=1, label="history")
    ax.plot(range(H), yte[i], color="k", lw=1.5, label="truth")
    ax.plot(range(H), forecast[i], color="crimson", lw=2, ls="--", label="forecast")
    ax.plot(
        range(H),
        level_part[i],
        color="tab:blue",
        lw=1,
        ls="--",
        label="level part (start + trend stack)",
    )
    ax.plot(
        range(H),
        seasonal_part[i],
        color="tab:green",
        lw=2,
        ls="--",
        label="seasonality stack",
    )
    ax.axvline(0, color="0.85")
    ax.set_xlabel("10-minute steps from the forecast start")
axes[0].set_ylabel("scaled solar power")
axes[-1].legend(fontsize=7)
fig.suptitle("the forecast is the level part plus the seasonality stack")
plt.tight_layout()

ratio = movement(seasonal_part) / movement(level_part)
print(f"movement within the horizon, level part      {movement(level_part):.4f}")
print(f"movement within the horizon, seasonality     {movement(seasonal_part):.4f}")
print(f"ratio seasonality / level                    {ratio:.3f}")

**Most of the movement within a day is the seasonality stack.** The level part (the start plus
the trend stack) moves 0.0735 within a horizon, on average; the seasonality stack moves
0.7752, about 10.5 times as much. In the plot the level part is a nearly flat line, and the seasonality stack draws the night-day-night shape on top of it. The
model states, in numbers you can read, which part of its forecast is "the level of today" and
which part is "it is 2pm".

The start has to be counted in: on its own the trend stack mostly cancels the start (the last
observed value is low when a window ends at night and high when it ends in daylight), so its
size says more about the time of day at the forecast start than about any trend.

On this series the interpretable model does not trade accuracy for that legibility: it beat
linear and is level with generic (3.4). That is not guaranteed; 3.6 asks what happens once the
series' own structure stops cooperating.

Keep this ratio. It is about to change.

## 3.6 The same architecture on a series with no short season

Daily sunspot counts have the opposite of solar power's spectrum: `acf(1d)` near 1,
`acf(27d)` real but weak (the Sun's ~27-day rotation, see `sunspot_daily.py`), and the
single largest peak in the whole spectrum sitting at the ~11-year solar cycle — far too
slow for any window this notebook could hold. The architecture built above
has a seasonality stack tuned to a 144-step period. There is nothing at 144 days for
it to find. Its harmonics do land near the real 27-day rotation (the fifth harmonic of 144
days lasts 28.8 days, the sixth 24), but no harmonic sits on it.

Run it unchanged, at the same `W=288, H=144` solar power was windowed at, and watch
the movement ratio. The free baselines now include a seasonal naive at 27 days, the period
this series does have, so the trained models face the strongest copy rule available.

In [ ]:
sun = load_sunspot_daily()
mu, sigma = (
    sun.values[sun.splits["train"]].mean(),
    sun.values[sun.splits["train"]].std(),
)
sunspot_windows = {}
for split in ("train", "val", "test"):
    values = (sun.values[sun.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, 6)]
    sunspot_windows[split] = (win[:, :W], win[:, W:])
print({k: len(v[0]) for k, v in sunspot_windows.items()})

sun_xte, sun_yte = sunspot_windows["test"]
sun_results = {
    "naive (repeat last)": free(naive, sunspot_windows),
    "seasonal naive (144d)": free(seasonal_naive, sunspot_windows),
    f"seasonal naive ({SUN_PERIOD}d)": free(
        lambda x: seasonal_naive(x, SUN_PERIOD), sunspot_windows
    ),
}
for name, make in (
    ("linear", Linear),
    ("N-BEATS generic", generic),
    ("N-BEATS interpretable", interpretable),
):
    r = fit(make, sunspot_windows)
    sun_results[name] = r
    if name == "N-BEATS interpretable":
        _, sun_level, sun_seasonal = parts_of(r["model"], sun_xte[:512])
        sun_ratio = movement(sun_seasonal) / movement(sun_level)

table(sun_results)
print(f"\nmovement ratio seasonality / level, sunspot:  {sun_ratio:.3f}")
print(f"                                  solar power: {ratio:.3f}")

In [ ]:
plot_curves(
    sun_results,
    trained,
    "sunspot: validation loss per epoch (solid), training loss (dotted); the dot is the epoch kept",
)

Two things move, and neither is a simple flip.

**The seasonality stack's share shrinks.** Within a horizon it moves 1.9 times as much as the
level part on sunspot, against 10.5 times on solar power. There is nothing at a 144-day period
for it to draw, and its nearest harmonics (28.8 and 24 days) miss the 27-day rotation. The
stack still outputs a periodic shape, because the architecture leaves it no other kind; whether
that shape helps, the scores below cannot tell.

**The interpretable model cannot be told apart from either neighbour.** On sunspot's test
split, generic N-BEATS scores 0.2912 ± 0.0018, linear 0.2999 ± 0.0002, and interpretable
0.2982 ± 0.0072. Interpretable sits 0.0017 from linear and 0.0070 from generic, both within
one of its own seed stds: the same size of gap this notebook called "level" on solar power.
Generic is ahead of linear by 0.0087, about five times its seed std; the test span is 537
overlapping windows from one decade, a spread the seed std does not include, so read it as a
lead on this decade. What differs between the two series is not one thing: the period,
but also the number of training windows (103,700 against 9,147), the noise, the
persistence and the time scale. And what differs between the two N-BEATS models is the basis,
but also their size (633,184 against 423,472 parameters). The stopping epochs show that no
model was cut off while still learning: each stopped where its own validation loss flattened.

**Headroom over naive is not the same size either.** On solar power, copying yesterday alone
cuts the naive baseline's error by 70% (0.220 against 0.737), and no trained model did better
than that copy on test. On sunspot, the 27-day copy (0.362) is barely better than repeating
the last value (0.367), and the best model cuts naive's error by 21% (0.291 against 0.367).
Sunspot's `acf(1d)` sits at 0.96 (`sunspot_daily.py`), so "repeat yesterday" is already most
of the answer; whether some other architecture would cut more of what is left, this notebook
does not test.

| | seasonality at this window's period | headroom over naive | what the test split shows |
|---|---|---|---|
| solar power | strong, near-deterministic (24h) | large: seasonal naive alone cuts 70% | both N-BEATS beat linear (8x the seed spread), interpretable level with generic; none beats seasonal naive |
| sunspot | absent at 144d (real period is 27d, and 11yr besides) | small: best model cuts 21% | generic ahead of linear (5x its seed spread); interpretable cannot be told apart from either |

**A basis is a bet about the data.** Where its period is real, it bought a readable
decomposition at no measurable cost in score. Where it is not, this notebook cannot say it
cost anything or bought anything: the interpretable model lands between linear and generic,
too close to both to rank. *Name the period you are betting on, check whether the series
actually has it, and expect the bet to pay only when it does* — and expect a wrong bet to show
up as a component that carries less of the forecast, not as an error.

## 3.7 A cycle that is not a season: sunspots

You have already met this series. MADS-DAV `03.2-statistics-of-time.ipynb` ran
`statsmodels.seasonal_decompose` on yearly sunspot counts and watched it fail: the
trend absorbed a cycle "of more than 100 years", the residual kept "very distinct
sine-wave patterns", and the notebook's own closing suggestions named "finding a way
to model the very long (+100 year) cycle" as unfinished business — handed forward on
purpose: *"In the course on Machine Learning we will go much deeper into this topic."*

This is that hand-off, with the basis classes you just built rather than a black-box
decomposition. Three named solar cycles sit in this record, at very different
timescales — the Schwabe cycle (~11yr), the Gleissberg cycle (~87yr, an amplitude
modulation of Schwabe), and the Suess/de Vries cycle (~210yr). The question is not
"does a fixed-period basis work here" — you already know a period must be roughly
fixed for that (3.6). It is: **how many of these three periods can 203 years of data
actually support?**


In [ ]:
def load_sunspot() -> torch.Tensor:
    # same rolling-eval convention as every Monash config: "test" holds the longest row.
    ds = load_dataset(
        "Monash-University/monash_tsf",
        revision="refs/convert/parquet",
        data_files={"test": "sunspot/test/0000.parquet"},
    )["test"][0]
    values = np.asarray(ds["target"], dtype="float32")
    dates = pd.date_range(start=ds["start"], periods=len(values), freq="D")
    yearly = pd.Series(values, index=dates).resample("YE").mean().dropna()
    return torch.tensor(yearly.to_numpy(), dtype=torch.float32), yearly.index


sunspot_yearly, sunspot_years = load_sunspot()
sun = (sunspot_yearly - sunspot_yearly.mean()) / sunspot_yearly.std()
N = len(sun)
print(
    f"{N} yearly means, {sunspot_years[0].year}-{sunspot_years[-1].year}  "
    f"({N / 11:.1f} Schwabe periods, {N / 87:.1f} Gleissberg periods, {N / 210:.2f} "
    "Suess/de Vries periods)"
)

This is not a forecasting window — 203 points is not enough to hold out a test span
and still fit an 87-year period, so this section fits the **whole series at once**, the
same thing `seasonal_decompose` did in DAV, using the exact `TrendBasis` and
`SeasonalityBasis` classes from 3.2 with `theta` solved directly (no `NBeatsBlock`,
no MLP — there is only one series, so there is nothing to generalise across). The periods
are set by hand: 11 years for Schwabe, 87 for Gleissberg. A score on the same 203 points
the fit saw is an in-sample score, so a second period can only be judged against controls
with the same number of shapes: one at the 11-year period with more harmonics, and a sweep
that tries every other second period from 20 to 300 years.

In [ ]:
def fit_decomposition(bases: list[nn.Module], target: torch.Tensor):
    # The backcast is linear in theta: its columns are the basis shapes. So the best theta
    # is an exact least-squares solve, no optimiser and no MLP: there is one series, and
    # nothing to generalise across.
    shapes = torch.cat([b.backcast_basis for b in bases]).T  # (N, number of shapes)
    theta = torch.linalg.lstsq(shapes, target.unsqueeze(1)).solution
    recon = (shapes @ theta).squeeze(1)
    r2 = 1 - float(
        ((recon - target) ** 2).sum() / ((target - target.mean()) ** 2).sum()
    )
    return recon, r2, shapes.shape[1]


def periodic(period: float, harmonics: int) -> SeasonalityBasis:
    return SeasonalityBasis(harmonics, period, backcast_size=N, forecast_size=1)


trend = TrendBasis(degree=1, backcast_size=N, forecast_size=1)
fits = {
    "trend + Schwabe(11yr)": [trend, periodic(11, 3)],
    "trend + Schwabe(11yr) + Gleissberg(87yr)": [
        trend,
        periodic(11, 3),
        periodic(87, 2),
    ],
    # same number of shapes as the line above, all at the 11-year period
    "trend + Schwabe(11yr), 5 harmonics [control]": [trend, periodic(11, 5)],
}
recon = {}
for name, bases in fits.items():
    recon[name], r2, k = fit_decomposition(bases, sun)
    print(f"{name:46s} R2 = {r2:.3f}   ({k} shapes)")

# the second slow component at every period from 20 to 300 years, same 12 shapes each time
periods = np.arange(20, 301, 1)
sweep = [
    fit_decomposition([trend, periodic(11, 3), periodic(int(q), 2)], sun)[1]
    for q in periods
]
best = int(periods[int(np.argmax(sweep))])
print(f"\nbest second period in the sweep: {best} years, R2 = {max(sweep):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3), gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(sunspot_years, sun, lw=0.6, color="0.6", label="yearly mean (scaled)")
for name in list(fits)[:2]:
    ax[0].plot(sunspot_years, recon[name], lw=1.2, label=name)
ax[0].legend(fontsize=8)
ax[0].set_ylabel("scaled sunspot number")
ax[0].set_title("fixed-period bases fitted to 203 years of sunspot counts")
ax[1].plot(periods, sweep, lw=1)
ax[1].axvline(87, color="crimson", ls="--", lw=1)
ax[1].text(87, min(sweep), " 87", color="crimson", fontsize=9)
ax[1].set_xlabel("period of the second slow component (years)")
ax[1].set_ylabel("in-sample R2")
ax[1].set_title("which second period fits best")
plt.tight_layout()

Adding the Gleissberg term raises the in-sample R² from 0.467 to 0.510, and the high-harmonic
control (12 shapes, all at the 11-year period) stays at 0.467. So the gain does not come from
having 12 shapes instead of 8. But the sweep shows it does not come from the 87-year period in
particular either: at 87 years R² sits on a rising slope, the curve peaks near 105 years
(about 0.57), dips around 150, and climbs to 0.579 at 300 years, the longest period tried. A
component longer than the record bends like a slow trend. What this fit supports is "a second,
slow component improves the in-sample fit over a high-harmonic control"; it does not pick out
87 years.

That is what 203 years can buy. The 11-year period here was set by hand, not estimated; 2.3
repeats of an 87-year cycle are too few for the data to single it out among its neighbours,
and the Suess/de Vries cycle at ~210 years is not even one full repeat in this record. That is
the concrete version of DAV's unfinished suggestion: not "the decomposition fails," but
"here is how much of it 203 years can support, and here is the line past which you would need
more data, not a better model."

## 3.8 How this differs from the reference implementation

Read
[`neuralforecast/models/nbeats.py`](https://github.com/Nixtla/neuralforecast/blob/main/neuralforecast/models/nbeats.py)
next to what you just wrote. `TrendBasis`, `SeasonalityBasis`, `IdentityBasis` and `NBEATSBlock`
are all plain `nn.Module`s taking plain tensors there too, and their stacking loop is the same
five lines. Two deliberate differences:

- **They reverse the window** (`insample_y.flip(dims=(-1,))`) so the most recent step comes first,
  and orient the bases to match. We keep forward time order. Self-consistent either way.
- **Their Fourier frequencies are derived from `forecast_size`**; ours are harmonics of a period
  *you name*. Ours is more readable and connects directly to the FFT bands you read off
  the spectrum, but it requires you to know the period — which the spectrum (3.1) is how you find out.

Their top-level `NBEATS` class subclasses a Lightning `BaseModel` and takes a `windows_batch`
dict, which is the only part that does not transfer. That is the general shape of these libraries:
the ideas are portable, the plumbing is not.

## What to take from this

- **A basis is a constraint on what the model may say**, and constraints are how you get
  interpretability. The stacks are trained together; the bases fix what kind of shape each one
  can emit, so the forecast always splits into a level part and a daily shape.
- **Designed features and designed *outputs* are not the same move.** Feed a trend slope in as an
  extra input and the model can ignore it, because a linear layer can already compute one.
  Here the trend is what the model is allowed to predict in, and it cannot be ignored.
- **Read the spectrum before you pick a period.** The seasonality stack is only as useful as the
  period you gave it, and section 3.6 shows what a wrong one buys you.
- **A basis is a bet about structure, not a substitute for data.** On solar power, whose
  144-step period is real, the interpretable model stayed level with generic N-BEATS and both
  beat linear by about eight times the seed spread. On sunspot, with no 144-step period, the
  interpretable model could not be told apart from linear or from generic (3.6).
- **Interpretability and accuracy are not always the same trade.** On solar power the model you
  can interrogate scored level with the one you cannot (interpretable 0.2571 vs generic 0.2508,
  a gap of about one seed std). Which one you ship is still a question about who has to defend
  the forecast, not something the leaderboard settles for you.
- **A baseline's score is a property of the period it is scored on, as much as of the
  baseline.** Seasonal naive is exactly as good as today is like yesterday. In November and
  December 2006 consecutive days were more alike than in January to March, and the copy rule's
  error fell with it while the models' error did not. A model that "loses" to seasonal naive on
  one autumn may have met easy weeks for the baseline, not hard weeks for itself. Before you read
  a gap to a baseline, look at how the baseline itself moves across the period; and one year
  cannot tell you whether easy weeks belong to the season or to that year's weather.
- **You still have to beat the cheap baseline.** On solar power's test split, copying
  yesterday (0.2201) beat every trained model in this notebook, though on validation the two
  N-BEATS models edged past it. Whatever the table said, it said it about this series, this horizon and this
  one autumn.

## Exercises

**E1 — another origin.** The test verdict in 3.4 comes from one autumn. Change the cuts in
`split_scale_window` to train on the first 40% of the year, validate on 40-60% (late May to
early August) and test on 60-80% (early August to mid October), leaving the rest out. Does any trained
model beat seasonal naive on that test span? Then set `PATIENCE` to 2 and to 10: do the best
epochs and the order on test change?

**E2 — the period is a hyperparameter.** The seasonality stack is fixed to a 144-step (24h)
period. Add a third stack at a week-scale period (1,008 steps) and re-measure. Solar power has
no real week-scale structure (the sun does not know what day it is) — does the extra stack pick
up anything, or does it sit near zero the way a stack tuned to the wrong period should?

**E3 — how much basis is enough?** Sweep `harmonics` over 1, 3, 6, 10, 16 and `degree` over 1, 2,
3. Plot score against the number of basis functions. Where does adding shapes stop helping, and
how does that compare with the band structure the FFT shows on this series?

**E4 — generic versus interpretable, on your own terms.** The generic model has an identity basis:
it can predict any shape. The interpretable one cannot. Measure both as you shrink the training
set (20,000, 5,000, 1,000 windows — pass `n_series` down to `load_solar` to do this cheaply).
Does N-BEATS's edge over linear grow, shrink, or flip as data gets scarcer?

**E5 — the search, before you see it.** `3_scaling_and_search/04_search` runs a search (`degree`, `harmonics`,
`hidden`, gate on/off) on `solar_10_minutes`. Before opening it, decide which of those axes
you expect to matter, the same way `1_representation_and_head/04_search` asked you to before its own
search ran.

**E6 — the honest comparison.** Add a PatchTST (`transformers.PatchTSTForPrediction`,
Nie et al., 2023) to the table on this series,
with the same seeds and the same early stopping. It has no notion of trend or seasonality and learns
everything from data. On a series this seasonal, does the inductive bias pay?